In [1]:
%%writefile /media/notebooks/EjercicioFinal/MonitorProceso.py
import subprocess
import time
import psutil


class MonitorProceso:

    def __init__(self, script):
        self.script = script
        self.tiempo = 0
        self.memoria_maxima = 0

    def ejecutar(self):

        inicio = time.perf_counter()

        proceso = subprocess.Popen(
            ["bash", self.script],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        proceso_psutil = psutil.Process(proceso.pid)

        while proceso.poll() is None:

            try:
                memoria_actual = (
                    proceso_psutil.memory_info().rss
                    / (1024 * 1024)
                )

                self.memoria_maxima = max(
                    self.memoria_maxima,
                    memoria_actual
                )

            except psutil.NoSuchProcess:
                break

            time.sleep(0.5)

        fin = time.perf_counter()

        self.tiempo = fin - inicio

        return {
            "script": self.script,
            "tiempo": self.tiempo,
            "memoria": self.memoria_maxima,
            "codigo_salida": proceso.returncode
        }

Overwriting /media/notebooks/EjercicioFinal/MonitorProceso.py


In [2]:
%%writefile /media/notebooks/EjercicioFinal/medir_procesos.py
from MonitorProceso import MonitorProceso


procesos = [
    ("MapReduce CSV", "mapreduce_csv.sh"),
    ("MapReduce Parquet", "mapreduce_parquet.sh"),
    ("MapReduce Avro", "mapreduce_avro.sh"),
    ("Hive CSV", "hive_csv.sh"),
    ("Hive Parquet", "hive_parquet.sh"),
    ("Hive Avro", "hive_avro.sh")
]


resultados = []


for nombre, script in procesos:

    print(f"\nEjecutando: {nombre}")

    monitor = MonitorProceso(script)

    resultado = monitor.ejecutar()

    resultados.append({
        "proceso": nombre,
        "tiempo": resultado["tiempo"],
        "memoria": resultado["memoria"],
        "codigo_salida": resultado["codigo_salida"]
    })

    print(
        f"Tiempo de ejecución: "
        f"{resultado['tiempo']:.2f} segundos"
    )

    print(
        f"Memoria máxima: "
        f"{resultado['memoria']:.2f} MB"
    )

    print(
        f"Código de salida: "
        f"{resultado['codigo_salida']}"
    )


Overwriting /media/notebooks/EjercicioFinal/medir_procesos.py


In [4]:
!cd EjercicioFinal/


In [ ]:
!python3 /media/notebooks/EjercicioFinal/medir_procesos.py


Ejecutando: MapReduce CSV
Tiempo de ejecución: 104.86 segundos
Memoria máxima: 3.38 MB
Código de salida: 0

Ejecutando: MapReduce Parquet
Tiempo de ejecución: 7.55 segundos
Memoria máxima: 3.39 MB
Código de salida: 0

Ejecutando: MapReduce Avro
Tiempo de ejecución: 13.96 segundos
Memoria máxima: 3.21 MB
Código de salida: 0

Ejecutando: Hive CSV
Tiempo de ejecución: 76.43 segundos
Memoria máxima: 3.41 MB
Código de salida: 0

Ejecutando: Hive Parquet
Tiempo de ejecución: 161.61 segundos
Memoria máxima: 3.28 MB
Código de salida: 0

Ejecutando: Hive Avro
Tiempo de ejecución: 176.68 segundos
Memoria máxima: 3.29 MB
Código de salida: 0
